# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassanbuilds/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal Check 1 — Staleness

I checked whether pages that have not been updated for a longer period show different search performance. This signal is linked to FlyRank's refresh logic.

**Verdict: MIXED**

The bucket results are mixed. Pages updated 91–180 days ago have higher median impressions than pages updated within 90 days, while the 181–365 and 365+ day groups have much lower median impressions. The very old groups also have small sample sizes, so this is only a directional signal.

In [15]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"]
)

staleness_check = df.groupby("staleness_bucket", observed=False).agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median")
)

print(staleness_check)

                      n  median_impressions
staleness_bucket                           
0-90 days         20655               472.0
91-180 days        9171              1692.0
181-365 days        169                16.0
365+ days             5                 2.0


### Signal Check 2 — Search Visibility

I checked how pages differ across search-impression levels. Impressions are the visibility signal used by my rule.

**Verdict: MIXED**

The relationship is weak and fairly similar across the impression buckets. Median days since update increases only slightly from 20 to 25 days across the non-empty groups. This suggests impressions alone do not strongly distinguish stale pages, so it should be treated as a supporting signal rather than a strong rule.

In [16]:
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 0, 100, 1000, 10000, np.inf],
    labels=["0", "1-100", "101-1k", "1k-10k", "10k+"]
)

visibility_check = df.groupby("visibility_bucket", observed=False).agg(
    n=("content_id", "size"),
    median_days_since_update=("days_since_last_update", "median")
)

print(visibility_check)

                      n  median_days_since_update
visibility_bucket                                
0                     0                       NaN
1-100              8006                      20.0
101-1k             8485                      22.0
1k-10k             9907                      22.0
10k+               3602                      25.0


### My rule

I will prioritize content pages for refresh review when they are stale and still have search visibility. The score will combine how long it has been since the page was updated with its recent search impressions.

### Reason codes

- `stale_visible` — the page has not been updated recently and still receives search impressions.
- `not_priority` — the page does not meet the conditions for the refresh-review rule.

### Action

- `review_refresh` — review the page and decide whether a content refresh is appropriate.
- `no_action` — do not prioritize the page for refresh review.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing seimport pandas as pd

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### My scoring rule

I will give higher scores to pages that have not been updated for a long time and still receive search impressions. Pages with higher scores will be prioritized for refresh review.

The rule uses only signals that would be available before making the refresh decision.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create the refresh score

import os

os.makedirs("work/outputs", exist_ok=True)

# Create the refresh score

df["staleness_score"] = df["days_since_last_update"].clip(lower=0)
df["visibility_score"] = np.log1p(df["impressions_90d"])

df["score"] = df["staleness_score"] * df["visibility_score"]

df["reason_code"] = np.where(
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] > 0),
    "stale_visible",
    "not_priority"
)

df["action"] = np.where(
    df["reason_code"] == "stale_visible",
    "review_refresh",
    "no_action"
)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked rows:", len(ranked))
print("CSV written: work/outputs/baseline_action_score.csv")



Ranked rows: 30000
CSV written: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

1. **content_cf56e2e2e282** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and has high impressions. It could be wrong if the page is already performing well for an important reason.

2. **content_7368877ea310** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and highly visible. It could be wrong if the page does not need updating despite its age.

3. **content_7f116ae1f6f5** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old but has relatively low impressions. It could be wrong if the low visibility is caused by factors unrelated to freshness.

4. **content_72496874f806** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old with low impressions. It could be wrong if refreshing the page would not improve its visibility.

5. **content_1bfaa38ff26c** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and has substantial impressions. It could be wrong if the page is already meeting its business goal.

6. **content_0a91db491d14** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and has meaningful impressions. It could be wrong if the content is still accurate and useful.

7. **content_6476d1d8c050** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old but has low impressions. It could be wrong if the page has limited opportunity regardless of freshness.

8. **content_4729b57ca036** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old with low impressions. It could be wrong if there is little search opportunity for the page.

9. **content_5feee3994adb** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and receives meaningful impressions. It could be wrong if the page does not need a content change.

10. **content_c2d929d83eaa** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and visible. It could be wrong if other factors are responsible for its performance.

11. **content_b16bd7307b39** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and receives impressions. It could be wrong if updating it would not change its performance.

12. **content_fe16a55cd13d** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and visible. It could be wrong if the content remains appropriate despite its age.

13. **content_ecb6215e79fd** — Action: review_refresh. Reason: stale_visible. Confidence: high because it is old and receives impressions. It could be wrong if freshness is not the main issue.

14. **content_df1fa766cac2** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old but has low impressions. It could be wrong if there is not enough search opportunity to justify a refresh.

15. **content_d25a099b3726** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old with low impressions. It could be wrong if refreshing the page would have little practical benefit.

16. **content_02b0d6e30129** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old but has low impressions. It could be wrong if the page has limited potential.

17. **content_f488400fca67** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old with low impressions. It could be wrong if the low visibility is unrelated to content freshness.

18. **content_cb7e312f5d32** — Action: no_action. Reason: not_priority. Confidence: low because it has high impressions but does not meet the stale threshold. It could be wrong if the page actually needs review for another reason not captured by this rule.

19. **content_928af3e22c80** — Action: review_refresh. Reason: stale_visible. Confidence: medium because it is old and has some impressions. It could be wrong if freshness is not limiting its performance.

20. **content_7a888d3d99c8** — Action: review_refresh. Reason: stale_visible. Confidence: low because it is very old but has very low impressions. It could be wrong if the page has little opportunity to benefit from a refresh.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked.head(20).copy()

top20[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]


,content_id,score,reason_code,action,days_since_last_update,impressions_90d
0,content_cf56e2e2e282,2139.761566,stale_visible,review_refresh,194,61678
1,content_7368877ea310,2132.695875,stale_visible,review_refresh,194,59472
2,content_7f116ae1f6f5,2065.375113,stale_visible,review_refresh,301,954
3,content_72496874f806,2020.233859,stale_visible,review_refresh,301,821
4,content_1bfaa38ff26c,1970.044517,stale_visible,review_refresh,194,25715
5,content_0a91db491d14,1832.635228,stale_visible,review_refresh,193,13299
6,content_6476d1d8c050,1790.457586,stale_visible,review_refresh,313,304
7,content_4729b57ca036,1750.950459,stale_visible,review_refresh,301,335
8,content_5feee3994adb,1738.927593,stale_visible,review_refresh,194,7812
9,content_c2d929d83eaa,1723.585378,stale_visible,review_refresh,193,7558


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

The weakest picks are the pages that receive a high score mainly because they are old but have very low impressions. These pages may not have enough search opportunity to justify a refresh.

I also observed one top-20 row with the `not_priority` reason code and `no_action`, showing that the ranking score and action condition are not perfectly aligned.

The rule does not use product flags, trend labels, trend percentages, or future-window information. It only uses `days_since_last_update` and `impressions_90d`, which are available signals for the decision. The ranking score and action condition are intentionally separate: a page can have a relatively high score but still receive `no_action` if it does not meet the stale-and-visible rule conditions.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show some of the weakest-looking picks from the top 20

weak_picks = ranked.head(20).sort_values(
    ["impressions_90d", "days_since_last_update"]
).head(5)

weak_picks[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

,content_id,score,reason_code,action,days_since_last_update,impressions_90d
19,content_7a888d3d99c8,1428.640984,stale_visible,review_refresh,313,95
16,content_f488400fca67,1540.206082,stale_visible,review_refresh,305,155
15,content_02b0d6e30129,1620.134866,stale_visible,review_refresh,313,176
14,content_d25a099b3726,1620.527824,stale_visible,review_refresh,305,202
13,content_df1fa766cac2,1621.146513,stale_visible,review_refresh,304,206


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.